In [1]:
# imports 

import subprocess
import selenium
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import pandas as pd
import time
import requests
import re
import csv
import os

In [2]:
# run at start of session only

# Check versions
result = subprocess.run(
    ["/Applications/Google Chrome.app/Contents/MacOS/Google Chrome", "--version"],
    capture_output=True, text=True
)
print(result.stdout)

print("Selenium version:", selenium.__version__)

options = Options()
# options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

driver = webdriver.Chrome(options=options)  # No Service() or ChromeDriverManager
driver.get("https://www.google.com")
print("Success! Title:", driver.title)
driver.quit()

Google Chrome 145.0.7632.160 

Selenium version: 4.41.0
Success! Title: Google


In [14]:
# Returns imdb reviews url for a given movie name with spaces, e.g. "The Batman"
def get_imdb_reviews_url(driver, movie_name):
    search_query = movie_name.replace(" ", "+")
    search_url = f"https://www.imdb.com/find/?q={search_query}&s=tt&ttype=ft"

    driver.get(search_url)
    time.sleep(5)  # Let the page fully load
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    first_result = soup.find("a", attrs={"class": "ipc-title-link-wrapper"})  # Updated class
    
    if not first_result:
        raise ValueError(f"No IMDB results found for: {movie_name}")
    
    href = first_result["href"]
    movie_id = href.split("/")[2]
    
    movie_url = f"https://www.imdb.com/title/{movie_id}/"
    reviews_url = f"https://www.imdb.com/title/{movie_id}/reviews/"
    return movie_url, reviews_url


# Scrapes movie information including movie title, iMDB rating, year, budget, and gross income across US and Canada
def get_movie_info_imdb(driver, main_url):
    print("Scraping movie info...")
    driver.get(main_url)
    time.sleep(5)

    print(f"Current Page Title: {driver.title}")
    
    if "Bot Check" in driver.title or "Just a moment" in driver.title:
        print("CRITICAL: IMDb is blocking the bot. Try turning off --headless.")
        
    soup = BeautifulSoup(driver.page_source, "html.parser")

    info = {}

    # Title
    tag = soup.find(attrs={"data-testid": "hero__primary-text"})
    info["title"] = tag.get_text(strip=True) if tag else "N/A"

    # Rating
    tag = soup.find(attrs={"data-testid": "hero-rating-bar__aggregate-rating__score"})
    info["imdb_rating"] = tag.get_text(strip=True).replace("/10", "") if tag else "N/A"

    # Year
    tag = soup.find(attrs={"data-testid": "hero-parent"})
    if tag:
        match = re.search(r'\b(19|20)\d{2}\b', tag.get_text())
        info["year"] = match.group(0) if match else "N/A"
    else:
        info["year"] = "N/A"

    # Budget
    tag = soup.find(attrs={"data-testid": "title-boxoffice-budget"})
    if tag:
        text = tag.get_text(strip=True).replace("Budget", "").replace("(estimated)", "").strip()
        info["budget"] = text
    else:
        info["budget"] = "N/A"

    # Gross US & Canada
    tag = soup.find(attrs={"data-testid": "title-boxoffice-grossdomestic"})
    if tag:
        text = tag.get_text(strip=True).replace("Gross US & Canada", "").strip()
        info["gross_us"] = text
    else:
        info["gross_us"] = "N/A"

    print(f"Movie info: {info}")
    return info



# Scrape a movie's top 50 featured reviews
def scrape_reviews_imdb(driver, reviews_url):
    print("Scraping reviews...")
    driver.get(reviews_url)
    time.sleep(5)

    # Click "Load More" twice to scrape 75 reviews in total then truncate to 50
    for i in range(2):
        try:
            load_more = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "button.ipc-see-more__button"))
            )
            driver.execute_script("arguments[0].scrollIntoView(true);", load_more)
            time.sleep(1)
            driver.execute_script("arguments[0].click();", load_more)
            time.sleep(3)
            print("Loaded more reviews...")
        except:
            print("All reviews loaded.")

    soup = BeautifulSoup(driver.page_source, "html.parser")
    reviews = []
    articles = soup.find_all("article", class_="user-review-item")
    print(f"Found {len(articles)} reviews, parsing...")

    count = 1

    for article in articles:
        review = {}

        review["id"] = f"{count}"
        count += 1

        # Rating — only present if user gave one
        rating_tag = article.find("span", class_="ipc-rating-star--rating")
        review["user_rating"] = rating_tag.get_text(strip=True) if rating_tag else "N/A"

        # Title
        title_tag = article.find("h3", class_="ipc-title__text")
        review["review_title"] = title_tag.get_text(strip=True) if title_tag else "N/A"

        # Review text
        text_tag = article.find("div", attrs={"data-testid": "review-overflow"})
        if not text_tag:
            text_tag = article.find("div", class_=lambda c: c and "content" in c.lower())
        review["review_text"] = text_tag.get_text(strip=True) if text_tag else "N/A"

        # Author
        author_tag = article.find("a", attrs={"data-testid": "author-link"})
        review["author"] = author_tag.get_text(strip=True) if author_tag else "N/A"

        # Date
        date_tag = article.find("li", attrs={"data-testid": "review-date"})
        review["date"] = date_tag.get_text(strip=True) if date_tag else "N/A"

        reviews.append(review)

    reviews = reviews[:50]
    return reviews


In [11]:
def get_imdb_reviews_url_api(movie_name):
    # Use IMDB's suggestion API — fast, no Selenium needed
    search_query = movie_name.replace(" ", "_")
    api_url = f"https://v3.sg.media-imdb.com/suggestion/x/{search_query}.json"
    
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    response = requests.get(api_url, headers=headers)
    data = response.json()
    
    # Filter for movies only (qid == "movie") and grab the first one
    movies = [r for r in data.get("d", []) if r.get("qid") == "movie"]
    
    if not movies:
        raise ValueError(f"No IMDB results found for: {movie_name}")
    
    movie_id = movies[0]["id"]  # e.g. tt6751668
    title = movies[0]["l"]

    # clean title for use in naming reviews csv
    title_clean = re.sub(r"[^\w]", "_", title.lower())
    title_clean = re.sub(r"_+", "_", title_clean).strip("_") 
    
    movie_url = f"https://www.imdb.com/title/{movie_id}/"
    reviews_url = f"https://www.imdb.com/title/{movie_id}/reviews/"
    print(f"Found: {title} → {movie_url}")
    print(f"Found: {title} Reviews → {reviews_url}")
    return movie_url, reviews_url, title_clean

def batch_process_movies(movie_list, output_folder):
    # Ensure folder exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Setup Selenium Driver once
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0...") 
    driver = webdriver.Chrome(options=options)

    master_metadata = []

    for movie_name in movie_list:
        print(f"\n--- Processing: {movie_name} ---")
        try:
            # 1. Get URLs
            movie_url, reviews_url, title_clean = get_imdb_reviews_url_api(movie_name)
            if not movie_url:
                print(f"Skipping {movie_name}: Could not find on IMDb.")
                continue
            
            movie_info = get_movie_info_imdb(driver, movie_url)
            movie_info['movie_key'] = title_clean # This links it to the review CSV filename
            master_metadata.append(movie_info)
            print(f"Metadata captured for {movie_name}")

            # 2. Get Reviews
            reviews = scrape_reviews_imdb(driver, reviews_url)
            
            # 3. Save to CSV
            if reviews:
                filepath = os.path.join(output_folder, f"{title_clean}_imdb_reviews.csv")
                with open(filepath, "w", newline="", encoding="utf-8") as f:
                    fieldnames = ["id", "author", "date", "user_rating", "review_title", "review_text"]
                    writer = csv.DictWriter(f, fieldnames=fieldnames)
                    writer.writeheader()
                    writer.writerows(reviews)
                print(f"Success! Saved {len(reviews)} reviews to {filepath}")
            else:
                print(f"No reviews found for {movie_name}")

            # 4. Small sleep to avoid getting blocked
            time.sleep(2)

        except Exception as e:
            print(f"An error occurred while processing {movie_name}: {e}")

    if master_metadata:
        metadata_df = pd.DataFrame(master_metadata)
        metadata_path = os.path.join(output_folder, "movie_metadata.csv")
        metadata_df.to_csv(metadata_path, index=False)
        print(f"\nSUCCESS: Master metadata saved to {metadata_path}")
 
    driver.quit()
    print("\nBatch Processing Complete.")

### main

In [12]:
if __name__ == "__main__":
    # This is where you put your actual list of 300 movies
    my_movies = ['The Shawshank Redemption',
                'Parasite',
                'Star Wars',
                'The Godfather',
                'Top Gun: Maverick',
                'The Godfather: Part II',
                'Spider-Man: Into the Spider-Verse',
                'Raiders of the Lost Ark',
                'The Dark Knight',
                'Mission: Impossible - Dead Reckoning',
                '12 Angry Men',
                'Pulp Fiction',
                'Schindler\'s List',
                'Mad Max: Fury Road',
                'Harakiri',
                'The Lord of the Rings: Return of the King',
                'Spirited Away',
                'Ikiru',
                'Inside Out',
                'The Godfather Part II',
                'LOTR: The Fellowship of the Ring',
                'Coco',
                'Goodfellas',
                'The Good, the Bad and the Ugly',
                'Toy Story 3',
                'Rear Window',
                'Forrest Gump',
                'Toy Story 4',
                'City Lights',
                'Fight Club',
                'Finding Nemo',
                'Alien',
                'LOTR: The Two Towers',
                'How to Train Your Dragon',
                'The Silence of the Lambs',
                'Inception',
                'Up',
                'The Shining',
                'Star Wars: Empire Strikes Back',
                'Star Trek (2009)',
                'Se7en',
                'The Matrix',
                'The Social Network',
                'The Apartment',
                'Eighth Grade',
                'Casablanca',
                'One Flew Over the Cuckoo\'s Nest',
                'Argo',
                'Seven Samurai',
                'Portrait of a Lady on Fire',
                'Hell or High Water',
                'It\'s a Wonderful Life',
                'Memento',
                'Psycho',
                'Spider-Man: No Way Home',
                'Singin\' in the Rain',
                'City of God',
                'The Farewell',
                'Apocalypse Now',
                'Saving Private Ryan',
                'Whiplash',
                'Vertigo',
                'Life Is Beautiful',
                'Your Name',
                'Citizen Kane',
                'The Green Mile',
                'The Pianist',
                'Taxi Driver',
                'Star Wars: A New Hope',
                'Harry Potter & Deathly Hallows Part 2',
                '2001: A Space Odyssey',
                'Interstellar',
                'Spider-Man: Across the Spider-Verse',
                'Dr. Strangelove',
                'Terminator 2: Judgment Day',
                'Get Out',
                'Double Indemnity',
                'Back to the Future',
                'Selma',
                'Raging Bull',
                'Moonlight',
                'Chinatown',
                'Lady Bird',
                'Lawrence of Arabia',
                'Dunkirk',
                'Sunset Boulevard',
                'Paddington 2',
                'Jaws',
                'Leon: The Professional',
                'Black Panther',
                'Some Like It Hot',
                'The Lion King',
                'In the Mood for Love',
                'Paths of Glory',
                'Gladiator',
                'Arrival',
                'American History X',
                'Inside Llewyn Davis',
                'Toy Story',
                'The Departed',
                'Marriage Story',
                'The Prestige',
                'Anatomy of a Fall',
                'The Third Man',
                'Boyhood',
                'The Wizard of Oz',
                'The Usual Suspects',
                'Gravity',
                'Roma',
                'Grave of the Fireflies',
                'The Holdovers',
                'Rashomon',
                'Everything Everywhere All at Once',
                'North by Northwest',
                'The Intouchables',
                'Minari',
                'Blade Runner',
                'Modern Times',
                'Past Lives',
                'Blue Velvet',
                'Once Upon a Time in the West',
                'Carol',
                '8½',
                'If Beale Street Could Talk',
                'On the Waterfront',
                'Cinema Paradiso',
                'First Cow',
                'Amadeus',
                'Call Me by Your Name',
                'Do the Right Thing',
                'Widows',
                'Ran',
                '12 Years a Slave',
                'Metropolis',
                'The Wolf of Wall Street',
                'Unforgiven',
                'Shoplifters',
                'Django Unchained',
                'Brooklyn',
                'Touch of Evil',
                'Won\'t You Be My Neighbor?',
                'The Lives of Others',
                'The Favourite',
                'Stalker',
                'Sunset Blvd.',
                'A Separation',
                'Annie Hall',
                'Little Women (2019)',
                'Bridge on the River Kwai',
                'Phantom Thread',
                'Great Expectations',
                'The Great Dictator',
                'Tár',
                'The General',
                'Witness for the Prosecution',
                'Before Sunset',
                'Butch Cassidy & Sundance Kid',
                'Avengers: Infinity War',
                'The Shape of Water',
                'To Kill a Mockingbird',
                'Aliens',
                'The Handmaiden',
                'Stagecoach',
                'American Beauty',
                'Manchester by the Sea',
                'All About Eve',
                'Once Upon a Time in Hollywood',
                'General',
                'Burning',
                'Fargo',
                'Oldboy',
                'Under the Skin',
                'The 400 Blows',
                'Princess Mononoke',
                'Her',
                'Dark Knight Rises',
                'No Country for Old Men',
                'Ben-Hur',
                'The Grand Budapest Hotel',
                'Bonnie and Clyde',
                'La La Land',
                'Beauty and the Beast (1946)',
                'Braveheart',
                'Birdman',
                'Breathless',
                'Joker',
                'Spotlight',
                'Bicycle Thieves',
                'Ex Machina',
                'O Brother, Where Art Thou?',
                'Notorious',
                'Das Boot',
                'Mulholland Drive',
                'The Searchers',
                'Inglourious Basterds',
                'Pan\'s Labyrinth',
                'Hamilton',
                'Amélie',
                'Come and See',
                '3 Idiots',
                'Million Dollar Baby',
                'The Bridge on the River Kwai',
                'Brokeback Mountain',
                'A Clockwork Orange',
                'The Sting',
                'The Master',
                'Wild Strawberries',
                'Good Will Hunting',
                'Blade Runner 2049',
                'High Noon',
                'Requiem for a Dream',
                'The Deer Hunter',
                'Inside Out 2',
                'Eternal Sunshine of the Spotless Mind',
                'Avengers: Endgame',
                'Manhattan',
                'Wall-E',
                'The Great Escape',
                'John Wick',
                'L.A. Confidential',
                'Full Metal Jacket',
                'The Zone of Interest',
                'Aguirre, the Wrath of God',
                'Decision to Leave',
                'Strangers on a Train',
                'The Maltese Falcon',
                'Poor Things',
                'Heat',
                'Reservoir Dogs',
                'Kill Bill: Vol 1',
                'Barry Lyndon',
                'The Iron Claw',
                'Platoon',
                'Godzilla Minus One',
                'Seven',
                'Hunt for the Wilderpeople',
                'Bottoms',
                'The Wild Bunch',
                '1917',
                'Oppenheimer',
                'Rebecca',
                'Talk to Me',
                'The Treasure of the Sierra Madre',
                'The Wrestler',
                'Dog Day Afternoon'] 
    
    # Set the folder where you want files to go
    save_path = '/Users/Diane/Desktop/PSYCH 186B/project/reviews/'
    
    # Run the big function
    batch_process_movies(my_movies, save_path)


--- Processing: The Shawshank Redemption ---
Found: The Shawshank Redemption → https://www.imdb.com/title/tt0111161/
Found: The Shawshank Redemption Reviews → https://www.imdb.com/title/tt0111161/reviews/
Scraping movie info...
Movie info: {'title': 'The Shawshank Redemption', 'imdb_rating': '9.3', 'year': 'N/A', 'budget': '$25,000,000', 'gross_us': '$28,767,189'}
Metadata captured for The Shawshank Redemption
Scraping reviews...


KeyboardInterrupt: 

In [3]:
# 1. SETUP THE DRIVER (Visible Mode)
options = Options()
# options.add_argument("--headless") # KEEP THIS COMMENTED OUT
options.add_argument("--window-size=1200,900")
options.add_experimental_option("excludeSwitches", ["enable-automation"])

driver = webdriver.Chrome(options=options)

# 2. LIST THE MOVIES YOU NEED
# Put the names of the movies you currently have review CSVs for here
movies_to_fix = ["Argo", "Memento", "Harakiri", "The Green Mile"] 

save_path = '/Users/Diane/Desktop/PSYCH 186B/project/reviews/'
metadata_file = os.path.join(save_path, "movie_metadata.csv")

# 3. THE LOOP
for movie_name in movies_to_fix:
    print(f"\n--- Processing: {movie_name} ---")
    try:
        # Step A: Get URLs (Using your original API function)
        movie_url, _, title_clean = get_imdb_reviews_url_api(movie_name)
        
        # Step B: Get Info (Using your original scraping function)
        # It will open the window so you can watch it load
        info = get_movie_info_imdb(driver, movie_url)
        
        if info['title'] != "N/A":
            info['movie_key'] = title_clean
            
            # Step C: Save immediately to the CSV
            df_new = pd.DataFrame([info])
            if not os.path.exists(metadata_file):
                df_new.to_csv(metadata_file, index=False)
            else:
                df_new.to_csv(metadata_file, mode='a', header=False, index=False)
            
            print(f"✅ Success! Saved {movie_name}")
        else:
            print(f"❌ Failed to get data for {movie_name} (Returned N/A)")

        # Step D: Human-like pause
        time.sleep(8) 
        
    except Exception as e:
        print(f"⚠️ Error with {movie_name}: {e}")

driver.quit()
print("\nDone! Check your movie_metadata.csv")


--- Processing: Argo ---
⚠️ Error with Argo: name 'get_imdb_reviews_url_api' is not defined

--- Processing: Memento ---
⚠️ Error with Memento: name 'get_imdb_reviews_url_api' is not defined

--- Processing: Harakiri ---
⚠️ Error with Harakiri: name 'get_imdb_reviews_url_api' is not defined

--- Processing: The Green Mile ---
⚠️ Error with The Green Mile: name 'get_imdb_reviews_url_api' is not defined

Done! Check your movie_metadata.csv


In [ ]:
# try:
#     driver.quit()
# except:
#     pass

# options = Options()
# options.add_argument("--headless")
# options.add_argument("--no-sandbox")
# options.add_argument("--disable-dev-shm-usage")
# options.add_argument("--window-size=1920,1080")
# options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# driver = webdriver.Chrome(options=options)

# movie_url, reviews_url, title_clean = get_imdb_reviews_url_api("Parasite")
# movie_info = get_movie_info_imdb(driver, movie_url)
# reviews = scrape_reviews_imdb(driver, reviews_url)

# driver.quit()

# print(f"\nScraped {len(reviews)} reviews total.")

# # Save to CSV
# filepath = f'/Users/Diane/Desktop/PSYCH 186B/project/reviews/{title_clean}_imdb_reviews.csv'
# with open(filepath, "w", newline="", encoding="utf-8") as f:
#     fieldnames = ["id", "author", "date", "user_rating", "review_title", "review_text"]
#     writer = csv.DictWriter(f, fieldnames=fieldnames)
#     writer.writeheader()
#     writer.writerows(reviews)

# print(f"Saved to: {filepath}")

In [5]:
import os
import time
import re
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup

# --- 1. THE HELPERS ---

def get_imdb_reviews_url_api(movie_name):
    # Cleans "the_batman" back to "The Batman" for searching
    search_query = movie_name.replace("_", " ")
    api_url = f"https://v3.sg.media-imdb.com/suggestion/x/{search_query}.json"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    try:
        response = requests.get(api_url, headers=headers, timeout=5)
        data = response.json()
        movies = [r for r in data.get("d", []) if r.get("qid") == "movie"]
        if not movies: return None, None
        movie_id = movies[0]["id"]
        movie_url = f"https://www.imdb.com/title/{movie_id}/"
        return movie_url, movies[0]["l"]
    except:
        return None, None

def get_movie_info_imdb(driver, main_url):
    driver.get(main_url)
    time.sleep(6) 
    soup = BeautifulSoup(driver.page_source, "html.parser")
    info = {}
    
    tag = soup.find(attrs={"data-testid": "hero__primary-text"})
    info["title"] = tag.get_text(strip=True) if tag else "N/A"

    tag = soup.find(attrs={"data-testid": "hero-rating-bar__aggregate-rating__score"})
    info["imdb_rating"] = tag.get_text(strip=True).split('/')[0] if tag else "N/A"

    year_tag = soup.find("a", href=lambda x: x and 'releaseinfo' in x)
    info["year"] = year_tag.get_text(strip=True) if year_tag else "N/A"

    budget_tag = soup.find("li", attrs={"data-testid": "title-boxoffice-budget"})
    info["budget"] = budget_tag.get_text(strip=True).replace("Budget", "").split('(')[0].strip() if budget_tag else "N/A"

    gross_tag = soup.find("li", attrs={"data-testid": "title-boxoffice-grossdomestic"})
    info["gross_us"] = gross_tag.get_text(strip=True).replace("Gross US & Canada", "").strip() if gross_tag else "N/A"
    return info

def start_driver():
    options = Options()
    options.add_argument("--window-size=1200,1000")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    return webdriver.Chrome(options=options)

# --- 2. THE MAIN ENGINE ---

save_path = '/Users/Diane/Desktop/PSYCH 186B/project/reviews/'
metadata_file = os.path.join(save_path, "movie_metadata.csv")

# A. Find all review files you've already scraped
review_files = [f for f in os.listdir(save_path) if f.endswith("_imdb_reviews.csv")]
movie_keys_to_fix = [f.replace("_imdb_reviews.csv", "") for f in review_files]

# B. Identify what's actually missing from the metadata CSV
if os.path.exists(metadata_file):
    existing_df = pd.read_csv(metadata_file)
    # Check if 'movie_key' column exists, if not, create it
    if 'movie_key' in existing_df.columns:
        done_keys = existing_df['movie_key'].tolist()
        movie_keys_to_fix = [m for m in movie_keys_to_fix if m not in done_keys]

print(f"Found {len(movie_keys_to_fix)} movies needing metadata.")

driver = start_driver()

for key in movie_keys_to_fix:
    print(f"\n--- Recovering: {key} ---")
    try:
        # Check if window was closed and restart if needed
        try:
            _ = driver.window_handles
        except:
            print("Browser was closed. Reopening...")
            driver = start_driver()

        # Get URL
        url, full_name = get_imdb_reviews_url_api(key)
        if not url:
            print(f"Skipping {key}: Not found.")
            continue
            
        # Scrape
        info = get_movie_info_imdb(driver, url)
        
        if info['title'] != "N/A":
            info['movie_key'] = key
            df_new = pd.DataFrame([info])
            df_new.to_csv(metadata_file, mode='a', header=not os.path.exists(metadata_file), index=False)
            print(f"✅ Success: {info['title']}")
        else:
            print(f"❌ Failed to parse {key}. Check the browser for a captcha!")

        time.sleep(8) # Politeness delay

    except Exception as e:
        print(f"⚠️ Error with {key}: {e}")

driver.quit()
print("\nBatch Metadata Recovery Finished.")

Found 54 movies needing metadata.

--- Recovering: memento ---
✅ Success: Memento

--- Recovering: star_wars_episode_iv_a_new_hope ---
⚠️ Error with star_wars_episode_iv_a_new_hope: HTTPConnectionPool(host='localhost', port=52886): Read timed out. (read timeout=120)

--- Recovering: 12_angry_men ---
✅ Success: 12 Angry Men

--- Recovering: harry_potter_and_the_deathly_hallows_part_2 ---
✅ Success: Harry Potter and the Deathly Hallows: Part 2

--- Recovering: singin_in_the_rain ---
✅ Success: Singin' in the Rain

--- Recovering: the_pianist ---
✅ Success: The Pianist

--- Recovering: the_shawshank_redemption ---
✅ Success: The Shawshank Redemption

--- Recovering: spirited_away ---
✅ Success: Spirited Away

--- Recovering: eighth_grade ---
✅ Success: Eighth Grade

--- Recovering: interstellar ---
✅ Success: Interstellar

--- Recovering: taxi_driver ---
✅ Success: Taxi Driver

--- Recovering: how_to_train_your_dragon ---
✅ Success: How to Train Your Dragon

--- Recovering: hell_or_high_w